# Chapter 10 · Evaluating an LLM application

A bank of 50 questions with reference answers, three versions of the assistant of chapters 5 and 6, and four ways of measuring: automatic key checking, similarity to the reference, a model as a judge and manual labels. Also, the confidence interval of a difference and error analysis.

**You need the GPU:** menu *Runtime* → *Change runtime type* → *T4 GPU* → *Save*. This is the heaviest notebook in the book: about 800 calls to the language models. On a desktop GPU it takes about thirteen minutes, and quite a bit longer in Colab. Without a GPU it is not practical.

The full explanation is in the PDF chapter. This notebook is the lab.

📘 The book on Amazon: [paperback](https://www.amazon.com/dp/B0HMX9N1KQ) · [Kindle](https://www.amazon.com/dp/B0HMX44VJ4)

## 1. Install, load the search models and the data

In [ ]:
!pip install -q rank_bm25

In [ ]:
"""Chapter 10: Evaluating an LLM application. On GPU (on CPU, very slow)."""
import json
import random
import re
import unicodedata
import urllib.request
from pathlib import Path

import numpy as np
import torch
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder, SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"        # the assistant's model
JUDGES = ["Qwen/Qwen2.5-1.5B-Instruct", "Qwen/Qwen3-1.7B"]
EMBEDDING_MODEL = "intfloat/multilingual-e5-base"
RERANKING_MODEL = "BAAI/bge-reranker-v2-m3"
DATA = Path("data")
REPOSITORY = "https://raw.githubusercontent.com/HernanDiaz/llm-engineering/main/en/"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"



def load_language(name):
    """A language model and its tokenizer, on the GPU if there is one."""
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float32)
    return tok, model.to(DEVICE).eval()


def download(name):
    """A file from the data folder; in Colab, it is downloaded from the repository."""
    path = DATA / name
    if not path.exists():
        DATA.mkdir(exist_ok=True)
        urllib.request.urlretrieve(REPOSITORY + "data/" + name, path)
    text = path.read_text(encoding="utf-8")
    return json.loads(text) if name.endswith(".json") else text

In [ ]:
embeddings = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)
reranker = CrossEncoder(RERANKING_MODEL, device=DEVICE)
bank = download("evaluation_bank.json")
print(len(bank), "questions. One of them:")
print(json.dumps(bank[13], ensure_ascii=False, indent=1))

## 2. The assistant of chapters 5 and 6

In [ ]:
def knowledge_base():
    """FAQ, terms and conditions and catalog."""
    fragments = [{"source": f"FAQ, {q['id']}",
                  "text": q["question"] + " " + q["answer"]}
                 for q in download("faq.json")["questions"]]
    for section in re.split(r"\n(?=## )", download("terms_and_conditions.md"))[1:]:
        title, text = section.strip().split("\n", 1)
        fragments.append({"source": "Terms and conditions, " + title[3:],
                          "text": text.strip()})
    for p in download("catalog.json"):
        fragments.append({"source": f"Catalog, {p['ref']}", "text": (
            f"{p['name']} (reference {p['ref']}), {p['category']}, "
            f"${p['price']}. {p['description']}")})
    return fragments


def words(text):
    """Words for BM25: lowercase, without accents, references kept whole."""
    text = "".join(c for c in unicodedata.normalize("NFD", text.lower())
                   if unicodedata.category(c) != "Mn")
    return re.findall(r"[a-z]{2}-\d{4}|[a-z0-9]{3,}", text)


class Searcher:
    """Embeddings, BM25 and reranking, as in chapter 6."""

    def __init__(self, fragments):
        self.fragments = fragments
        self.texts = [f["text"] for f in fragments]
        self.matrix = embeddings.encode(["passage: " + t for t in self.texts],
                                        normalize_embeddings=True)
        self.bm25 = BM25Okapi([words(t) for t in self.texts])

    def semantic(self, query):
        """Every fragment, ordered by the embedding similarity, as in chapter 5."""
        v = embeddings.encode(["query: " + query], normalize_embeddings=True)[0]
        return list(np.argsort(-(self.matrix @ v)))

    def improved(self, query, n=20):
        """Hybrid search fused with RRF, then the first n candidates reranked."""
        scores = self.bm25.get_scores(words(query))
        bm25 = [i for i in np.argsort(-scores, kind="stable") if scores[i] > 0]
        rrf = {}
        for ranking in (self.semantic(query), bm25):
            for pos, i in enumerate(ranking, 1):
                rrf[i] = rrf.get(i, 0) + 1 / (60 + pos)
        candidates = sorted(rrf, key=rrf.get, reverse=True)[:n]
        marks = reranker.predict([(query, self.texts[i]) for i in candidates])
        return [candidates[i] for i in np.argsort(-marks)]

In [ ]:
DONT_KNOW = "I don't have that information; I'm passing your question to the team."


SYSTEM_RAG = (
    "You are the customer service assistant of North Cycles, a bicycle store in "
    "Portland, Oregon. Answer the customer's question in English, in two or three "
    "sentences, using only the information in the documents you are given. After "
    "each fact, give the number of the document it comes from in square brackets, "
    "for example [2]. If the documents do not contain the answer, reply exactly: "
    f"\"{DONT_KNOW}\" Do not make anything up.")


SYSTEM_CHECK = (
    "Decide whether the documents have anything to do with the customer's "
    "question. Answer \"no\" only if no document deals with the topic of the "
    "question; otherwise, answer \"yes\".")


def rag_prompt(question, documents):
    """The numbered documents and the question, as in chapter 5."""
    listing = "\n\n".join(f"[{i}] ({d['source']}) {d['text']}"
                          for i, d in enumerate(documents, 1))
    return f"Documents:\n\n{listing}\n\nCustomer's question: {question}"


RAG_EXAMPLES = [
    (rag_prompt("Do you have helmets for my five-year-old son?", [
        {"source": "Catalog, helmets", "text": "Kids' helmets in sizes XS "
         "(18-19.5 in) and S (19.5-21 in), recommended for ages 3 to 8."},
        {"source": "Terms and conditions, 3. Prices",
         "text": "Prices are in US dollars; North Cycles does not charge sales tax."}]),
     "Yes, we have kids' helmets in sizes XS and S, recommended for ages 3 to 8 "
     "[1]."),
    (rag_prompt("Do you sell electric scooters?", [
        {"source": "Catalog, e-bikes", "text": "City and mountain e-bikes, "
         "with a range of 40 to 75 miles."},
        {"source": "FAQ, 99", "text": "Do you rent bikes? "
         "Yes, city bikes and e-bikes by the day."}]),
     DONT_KNOW),
]


def generate(tok, model, system, user, examples=(), max_tokens=200):
    """A model's answer, with optional examples and temperature 0."""
    messages = [{"role": "system", "content": system}]
    for q, a in examples:
        messages += [{"role": "user", "content": q},
                     {"role": "assistant", "content": a}]
    messages.append({"role": "user", "content": user})
    text = tok.apply_chat_template(messages, tokenize=False,
                                   add_generation_prompt=True,
                                   enable_thinking=False)
    inputs = tok(text, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_tokens,
                                do_sample=False, temperature=None, top_p=None,
                                top_k=None, pad_token_id=tok.eos_token_id)
    answer = tok.decode(output[0, inputs["input_ids"].shape[1]:],
                        skip_special_tokens=True)
    return re.sub(r"<think>.*?</think>", "", answer, flags=re.S).strip()


def assistant(question, retrieve, check=False, k=3):
    """The assistant with one way of retrieving and, if asked, the check first
    of chapter 5 (in its least strict version)."""
    documents = [searcher.fragments[i] for i in retrieve(question)[:k]]
    prompt = rag_prompt(question, documents)
    if check and generate(tok, model, SYSTEM_CHECK, prompt,
                          max_tokens=3).lower().startswith("no"):
        return DONT_KNOW, documents
    return generate(tok, model, SYSTEM_RAG, prompt, RAG_EXAMPLES), documents

## 3. Four ways of measuring

In [ ]:
def contains(text, keys):
    """Does the text contain any of the keys? Curly apostrophes count as straight."""
    text = text.lower().replace("’", "'")
    return any(k.lower() in text for k in keys)


def automatic_check(case, answer):
    """Correct if it contains a key fact without abstaining, or if it abstains
    when the question has no answer."""
    abstains = contains(answer, [DONT_KNOW[:30]])
    if case["keys"] is None:
        return abstains
    return contains(answer, case["keys"]) and not abstains


def similarity(answer, reference):
    """Cosine similarity between the embeddings of the answer and the reference."""
    v = embeddings.encode(["query: " + answer, "query: " + reference],
                          normalize_embeddings=True)
    return float(v[0] @ v[1])


JUDGE_CORRECTNESS = (
    "You are a strict grader. Compare an assistant's answer with the reference "
    "answer, which is the correct one. The answer is correct if it contains the "
    "essential information of the reference and neither contradicts it nor adds "
    "false facts; style and length do not matter. If the reference says there is "
    "no information, the answer is only correct if it gives none either. Reply "
    "only \"correct\" or \"incorrect\".")


JUDGE_FAITHFULNESS = (
    "You are a strict grader. Decide whether everything the answer states is "
    "supported by the documents. If the answer says it does not have the "
    "information, it counts as supported. Reply only \"yes\" or \"no\".")


def judge(tok_j, judge_model, case, answer, documents):
    """The judge's two marks: correctness against the reference and
    faithfulness to the documents."""
    text = (f"Question: {case['question']}\n\nReference: {case['reference']}"
            f"\n\nAssistant's answer: {answer}")
    correct = generate(tok_j, judge_model, JUDGE_CORRECTNESS, text, max_tokens=4)
    docs = "\n".join(f"- {d['text']}" for d in documents)
    faithful = generate(tok_j, judge_model, JUDGE_FAITHFULNESS,
                        f"Documents:\n{docs}\n\nAnswer: {answer}", max_tokens=3)
    return {"correct": correct.lower().lstrip("\"'*").startswith("correct"),
            "faithful": faithful.lower().lstrip("\"'*").startswith("yes")}

## 4. Is the difference real?

In [ ]:
def paired_difference(a, b, repetitions=10000, seed=0):
    """Difference in correct answers between two systems on the same questions,
    with a 95% confidence interval by resampling (bootstrap)."""
    rng = random.Random(seed)
    n = len(a)
    diffs = []
    for _ in range(repetitions):
        sample = [rng.randrange(n) for _ in range(n)]
        diffs.append(sum(b[i] - a[i] for i in sample))
    diffs.sort()
    return {"difference": sum(b) - sum(a),
            "interval": (diffs[int(0.025 * repetitions)],
                         diffs[int(0.975 * repetitions)]),
            "only_a": sum(x and not y for x, y in zip(a, b)),
            "only_b": sum(y and not x for x, y in zip(a, b))}


def agreement(automatic, manual):
    """Share of cases in which a measure agrees with the manual labels."""
    return sum(x == y for x, y in zip(automatic, manual)) / len(manual)

In [ ]:
# An example with made-up data: two systems that get 30 and 33 out of 50 right.
import random
rng = random.Random(1)
a = [rng.random() < 0.6 for _ in range(50)]
b = [x or rng.random() < 0.15 for x in a]
print(sum(a), sum(b), paired_difference(a, b))

## 5. Evaluate the three systems

This is the slow part.

In [ ]:
SYSTEMS = {
    "A. Chapter 5 RAG": dict(retrieve="semantic", check=False),
    "B. Chapter 6 retrieval": dict(retrieve="improved", check=False),
    "C. B + check first": dict(retrieve="improved", check=True),
}


def run():
    """Answers the bank with the three systems, judges them and compares them."""
    global searcher, tok, model
    bank = download("evaluation_bank.json")
    searcher = Searcher(knowledge_base())
    tok, model = load_language(MODEL)
    results = {}
    for name, options in SYSTEMS.items():
        retrieve = getattr(searcher, options["retrieve"])
        rows = []
        for case in bank:
            answer, docs = assistant(case["question"], retrieve,
                                     options["check"])
            rows.append({
                "question": case["question"], "type": case["type"],
                "answer": answer, "documents": docs,
                "automatic": automatic_check(case, answer),
                "retrieved": case["keys"] is None or any(
                    contains(d["text"], case["keys"]) for d in docs),
                "similarity": similarity(answer, case["reference"])})
        results[name] = rows
        print(f"{name}: {sum(r['automatic'] for r in rows)}/50 correct")
    del model
    for judge_name in JUDGES:
        tok_j, judge_model = load_language(judge_name)
        for rows in results.values():
            for case, r in zip(bank, rows):
                r[judge_name] = judge(tok_j, judge_model, case, r["answer"],
                                      r["documents"])
        del judge_model
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
    for r in (r for rows in results.values() for r in rows):
        r["documents"] = [d["source"] for d in r["documents"]]
    a, b = (results[n] for n in list(SYSTEMS)[:2])
    results["A vs B"] = paired_difference(
        [r["automatic"] for r in a], [r["automatic"] for r in b])
    print(results["A vs B"])
    return results

In [ ]:
results = run()

## 6. Compare with the manual labels

The manual labels belong to the answers of a specific run; if yours change, review and correct them. If the labels file is not available, this part is skipped.

In [ ]:
def manual_labels(path):
    """The manual labels of system B (1 correct, 0 wrong), or None if the file
    is missing."""
    path = Path(path)
    if not path.exists():
        print(f"No manual labels ({path.name}): agreement with them is skipped.")
        return None
    return json.loads(path.read_text(encoding="utf-8"))["correct"]


def compare_with_labels(results, manual):
    """Agreement of each measure with the manual labels, on system B."""
    if manual is None:
        return
    rows = results["B. Chapter 6 retrieval"]
    measures = {"Automatic check": [r["automatic"] for r in rows],
                "Similarity >= 0.85": [r["similarity"] >= 0.85 for r in rows]}
    for judge_name in JUDGES:
        measures[f"Judge {judge_name}"] = [r[judge_name]["correct"] for r in rows]
    for name, v in measures.items():
        print(f"{name:38s} {sum(v):2d} correct, "
              f"agrees in {agreement(v, manual):.0%}")

In [ ]:
path = Path("chapter_10/manual_labels.json")   # in the repository, from en/
if not path.exists():                           # in Colab, it is downloaded
    path = Path("manual_labels.json")
    try:
        urllib.request.urlretrieve(REPOSITORY + "chapter_10/manual_labels.json", path)
    except OSError:
        pass
manual = manual_labels(path)
compare_with_labels(results, manual)

The errors of system B, according to the manual labels, with whether the correct fragment was retrieved:

In [ ]:
if manual is not None:
    B = results["B. Chapter 6 retrieval"]
    for case, r, ok in zip(bank, B, manual):
        if not ok:
            print(f"[{case['type']}] {case['question']}")
            print("    retrieved:", r["retrieved"], "| answer:", r["answer"][:150])

## Exercises

1. Explain in your own words the difference between correctness and faithfulness. Give an example of an answer that is correct but not faithful, and another of an answer that is faithful but incorrect.

2. Label by hand the 50 answers of system A with the same criterion as the chapter. How many are correct? Does your comparison of A and B match that of the automatic check?

3. Review the chapter's manual labels. Do you agree with all of them? Change the ones that don't convince you and recompute the agreement of each measure.

4. Improve the prompt of the Qwen3-1.7B judge to make it stricter, for example with two worked examples of incorrect answers. Does its agreement with the labels improve?

5. Turn on the reasoning of the Qwen3-1.7B judge (`enable_thinking=True`, with more output tokens). Does it judge better? How much longer does it take?

6. Compute the confidence interval of the difference between B and C. Can you claim that the check first improves the system?

7. Double the test bank by copying the 50 questions, and compute the interval between A and B again. Why is it misleading? What would you really have to do to get a bigger bank?

8. Use the faithfulness judge to find the answers of system B that state something that is not in the documents. Do they match the made-up answers you found by hand?

9. Add ten new questions to the bank, of the types where the system fails most, with their reference and their keys. Does the error analysis change?

10. **Evaluation report.** Write a one-page report on the assistant: test bank, measures used and how reliable they are, results of the three systems, error analysis and recommendations for the next version.

## References

Models [Qwen2.5-1.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct), [Qwen3-1.7B](https://huggingface.co/Qwen/Qwen3-1.7B), [multilingual-e5-base](https://huggingface.co/intfloat/multilingual-e5-base) and [bge-reranker-v2-m3](https://huggingface.co/BAAI/bge-reranker-v2-m3). Model as a judge: [Zheng et al. (2023)](https://arxiv.org/abs/2306.05685). [RAGAS](https://arxiv.org/abs/2309.15217) (Es et al., 2023). The North Cycles store and its documents are fictional. Original code and experiments.